# AOS SFT — Qwen2.5-Coder-7B-Instruct (Colab)

Fine-tune **Qwen2.5-Coder-7B-Instruct** on Manim Code Agent trajectories (QLoRA + TRL).

**Before running:** add Colab secrets `HF_TOKEN` and (optional) `WANDB_API_KEY` via the key icon in the left sidebar.

See `apps/sft/docs/MODEL_SELECTION.md` for model rationale.

In [ ]:
%cd /content
!nvidia-smi

In [ ]:
# Clone (or refresh) and use the Qwen SFT branch
import os

BRANCH = "feat/qwen25-coder-7b-sft"
REPO = "https://github.com/nabin2004/AOS.git"

if not os.path.isdir("/content/AOS/.git"):
    !git clone {REPO} /content/AOS
%cd /content/AOS
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}

In [ ]:
import os
from google.colab import userdata

os.environ["UV_LINK_MODE"] = "copy"

# Required: Colab secret HF_TOKEN
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Optional W&B
try:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    os.environ.setdefault("WANDB_ENTITY", userdata.get("WANDB_ENTITY"))
except Exception:
    print("WANDB_API_KEY not set — training will use --report-to none if needed")

os.environ["WANDB_PROJECT_SFT"] = "aos-sft"
os.environ["WANDB_RUN_NAME"] = "qwen25-coder-7b-manim-sft"

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])
print("HF login OK; base model = Qwen/Qwen2.5-Coder-7B-Instruct")

In [ ]:
%cd /content/AOS
!uv sync --package sft

## Preflight (template + assistant mask)

Run before a long job. Confirms `<tool_call>` / `<tool_response>` render and that tool errors are **not** in the loss mask.

In [ ]:
%cd /content/AOS
!uv run --package sft python apps/sft/preflight_sft.py --colab --report-to none

## Train (QLoRA)

`--colab` preset: batch 1, seq 4096, packing off, output under Google Drive when mounted.

Default output: `/content/drive/MyDrive/qwen25-coder-7b-manim-ft` (or `/content/qwen25-coder-7b-manim-ft` if Drive is not mounted).

In [ ]:
%cd /content/AOS
# Mount Drive first if you want a durable checkpoint:
# from google.colab import drive; drive.mount("/content/drive")

!uv run --package sft python apps/sft/run.py --colab --epochs 2 --report-to wandb

## Infer (tool loop)

In [ ]:
%cd /content/AOS
!uv run --package sft python apps/sft/infer.py \
  --adapter-dir /content/qwen25-coder-7b-manim-ft \
  --colab \
  --prompt "Create a short Manim scene explaining eigenvectors in 2D."

## Upload LoRA adapter to Hugging Face

In [ ]:
%cd /content/AOS
!uv run --package sft python apps/sft/upload_adapter.py \
  --adapter-dir /content/qwen25-coder-7b-manim-ft \
  --colab

## Merge adapter → bf16 (optional)

In [ ]:
%cd /content/AOS/apps/sft
!uv run python merge_adapter.py \
  --adapter-dir /content/qwen25-coder-7b-manim-ft \
  --output-dir /content/qwen25-coder-7b-manim-merged \
  --push-to-hub

## GGUF export (optional)

Needs a recent [llama.cpp](https://github.com/ggml-org/llama.cpp) build.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cd llama.cpp && cmake -B build && cmake --build build -j --config Release

In [ ]:
import os

os.environ["LLAMA_CPP_DIR"] = "/content/llama.cpp"
%cd /content/AOS/apps/sft
!uv run python export_gguf.py \
  --model-dir /content/qwen25-coder-7b-manim-merged \
  --output-dir /content/qwen25-coder-7b-manim-gguf \
  --push-to-hub \
  --skip-ollama-create

In [ ]:
print("DONE — Qwen2.5-Coder-7B Manim SFT notebook finished.")